# HopSkipJump Attack on All Models

This notebook assumes you already have the following helper functions available
in the environment (import them from your own modules before running this):

- `run_logreg("CSVs/newDataset.csv")`
- `run_neuralnet("CSVs/newDataset.csv")`
- `run_randomforest("CSVs/newDataset.csv")`
- `run_svm("CSVs/newDataset.csv")`
- `run_xgboost("CSVs/newDataset.csv")`

Each function is expected to return a tuple:

```python
(model, X_test, y_test)
```

where:
- `model` is a trained classifier (pipeline or estimator)
- `X_test` is the test feature matrix (pandas DataFrame or numpy array)
- `y_test` is the corresponding true labels

The notebook then applies the HopSkipJump attack from ART to each model and
reports clean vs adversarial accuracy.


In [6]:
# Imports

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score

# Adversarial Robustness Toolbox (ART)
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump

# IMPORTANT:
# Make sure you import your run_* helpers here, for example:
#
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost
#
# or import your wrapper functions that already call those.


import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)



In [7]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)

In [8]:
# Run all models on the same dataset path
# Assumes each function returns (model, X_test, y_test)

DATA_PATH = "CSVs/newDataset.csv"

model_runs = {}

print("Running Logistic Regression...")
logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

print("Running Neural Net...")
nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

print("Running Random Forest...")
rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

print("Running SVM...")
svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

print("Running XGBoost...")
xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

print("\nSummary of collected models:")
for name, (model, X_test, y_test) in model_runs.items():
    print(f" - {name}: model={type(model)}, X_test shape={getattr(X_test, 'shape', None)}")



Running Logistic Regression...
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.936)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1268      84
True 1      58     289

AUC: 0.936

=== All results and summaries saved successfully ===
Running Neural Net...
Using parameters: {'hidden_layer_sizes': '128,64', 'alpha': 0.0001, 'learning_rate_

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [18:33:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [16]:
# HopSkipJump attack on each model

import xgboost as xgb
from art.estimators.classification import SklearnClassifier, XGBoostClassifier
from art.attacks.evasion import HopSkipJump
import numpy as np
import pandas as pd

hsj_kwargs = dict(
    max_iter=20,
    max_eval=10000,
    init_eval=100,
    init_size=10,
    targeted=False,
    norm=2,
)

results_hsj = {}

# Limit samples per model (keep or change to match your notebook)
MAX_SAMPLES = 200


def make_art_classifier(model, X, y, clip_values=(0, 1)):
    """
    Wraps the model in the correct ART classifier.
    Uses XGBoostClassifier for XGBoost models, SklearnClassifier otherwise.
    """
    # XGBoost branch
    if isinstance(model, xgb.XGBClassifier):
        print("Wrapping model with ART XGBoostClassifier for HopSkipJump")
        nb_classes = int(len(np.unique(y)))
        nb_features = int(X.shape[1])
        return XGBoostClassifier(
            model=model,
            nb_features=nb_features,
            nb_classes=nb_classes,
            clip_values=clip_values,
        )

    # Default: sklearn-compatible models
    print("Wrapping model with ART SklearnClassifier for HopSkipJump")
    return SklearnClassifier(model=model, clip_values=clip_values)


for name, (model, X_test, y_test) in model_runs.items():
    print("\n=== HopSkipJump attack on model:", name, "===")

    # Convert data to numpy
    if hasattr(X_test, "to_numpy"):
        X_all = X_test.to_numpy().astype(np.float32)
    else:
        X_all = np.asarray(X_test, dtype=np.float32)

    if hasattr(y_test, "to_numpy"):
        y_all = y_test.to_numpy()
    else:
        y_all = np.asarray(y_test)

    # Optional: limit number of samples
    n_total = len(X_all)
    n = min(MAX_SAMPLES, n_total)
    X = X_all[:n]
    y = y_all[:n]

    # Ensure labels are integer-encoded 1D
    if isinstance(y, pd.Series):
        y = y.to_numpy()
    y_int = y.astype(int).reshape(-1)

    n = X.shape[0]
    print(f"Using {n} samples for HopSkipJump on {name}")

    # Build ART classifier (XGBoost or Sklearn)
    art_classifier = make_art_classifier(model, X, y_int, clip_values=(0, 1))

    # Create HopSkipJump instance
    hsj_attack = HopSkipJump(classifier=art_classifier, **hsj_kwargs)

    clean_correct = 0
    adv_correct = 0

    for i in range(n):
        xi = X[i:i+1]
        yi = y_int[i:i+1]

        # Prediction on clean input
        pred_clean = np.argmax(art_classifier.predict(xi), axis=1)

        # Generate adversarial example (HopSkipJump is decision-based)
        x_adv = hsj_attack.generate(x=xi, y=yi)

        # Prediction on adversarial input
        pred_adv = np.argmax(art_classifier.predict(x_adv), axis=1)

        # Compare scalars explicitly
        clean_correct += int(pred_clean[0] == yi[0])
        adv_correct   += int(pred_adv[0]   == yi[0])

    clean_acc = clean_correct / n
    adv_acc = adv_correct / n

    print(f"{name} clean acc (HSJ): {clean_acc:.4f}")
    print(f"{name} adv   acc (HSJ): {adv_acc:.4f}")

    results_hsj[name] = dict(clean_acc=float(clean_acc), adv_acc=float(adv_acc))

hsj_results_df = pd.DataFrame(results_hsj).T
hsj_results_df



=== HopSkipJump attack on model: LogisticRegression ===
Using 200 samples for HopSkipJump on LogisticRegression
Wrapping model with ART SklearnClassifier for HopSkipJump


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 499.86it/s]


LogisticRegression clean acc (HSJ): 0.9100
LogisticRegression adv   acc (HSJ): 0.6750

=== HopSkipJump attack on model: NeuralNet ===
Using 200 samples for HopSkipJump on NeuralNet
Wrapping model with ART SklearnClassifier for HopSkipJump


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 10.61it/s]


NeuralNet clean acc (HSJ): 0.9600
NeuralNet adv   acc (HSJ): 0.7350

=== HopSkipJump attack on model: RandomForest ===
Using 200 samples for HopSkipJump on RandomForest
Wrapping model with ART SklearnClassifier for HopSkipJump


...

HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 496.19it/s]


SVM clean acc (HSJ): 0.9000
SVM adv   acc (HSJ): 0.1700

=== HopSkipJump attack on model: XGBoost ===
Using 200 samples for HopSkipJump on XGBoost
Wrapping model with ART XGBoostClassifier for HopSkipJump


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 11.05it/s]

XGBoost clean acc (HSJ): 0.9650
XGBoost adv   acc (HSJ): 0.2150


,clean_acc,adv_acc
LogisticRegression,0.910,0.675
NeuralNet,0.960,0.735
RandomForest,0.960,0.310
SVM,0.900,0.170
XGBoost,0.965,0.215


In [17]:
# ================================================================
#   SKLEARN METRICS SUMMARY FOR CLEAN + HOPSKIPJUMP PERFORMANCE
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# Ensure we have model_runs; rebuild if necessary
# ----------------------------------------------------------------
if "model_runs" not in globals():
    print("model_runs not found, rebuilding using run_* helpers...")

    DATA_PATH = "CSVs/newDataset.csv"
    model_runs = {}

    print("Running Logistic Regression...")
    logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
    model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

    print("Running Neural Net...")
    nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
    model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

    print("Running Random Forest...")
    rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
    model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

    print("Running SVM...")
    svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
    model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

    print("Running XGBoost...")
    xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
    model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

hsj_summary_rows = []

print("\n\n===================== HOPSKIPJUMP METRICS SUMMARY =====================\n")

for name, (model, X_test, y_test) in model_runs.items():

    print("\n\n################################################################")
    print("MODEL:", name)
    print("################################################################")

    # Convert labels
    if hasattr(y_test, "to_numpy"):
        y_true = y_test.to_numpy()
    else:
        y_true = np.asarray(y_test)

    # Clean predictions from the trained model
    y_pred = model.predict(X_test)

    # Basic clean metrics
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print("\nCLEAN PERFORMANCE")
    print("----------------------------")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # Row for summary table
    row = {
        "Model": name,
        "Clean Accuracy": acc,
        "Clean F1": f1
    }

    # ============================
    #   HOPSKIPJUMP RESULTS
    # ============================
    if "results_hsj" in globals() and name in results_hsj:
        hsj_clean = results_hsj[name]["clean_acc"]
        hsj_adv   = results_hsj[name]["adv_acc"]
        hsj_drop  = hsj_clean - hsj_adv

        print("\nHOPSKIPJUMP ADVERSARIAL RESULTS")
        print("----------------------------")
        print("Clean Accuracy (HSJ dict):", hsj_clean)
        print("Adv Accuracy   (HSJ):     ", hsj_adv)
        print("Accuracy Drop:", hsj_drop)

        row["HSJ Adv Accuracy"] = hsj_adv
        row["HSJ Drop"] = hsj_drop
    else:
        print("\nNo HSJ results recorded for this model in results_hsj.")

    hsj_summary_rows.append(row)

# Create dataframe summary
hsj_metrics_summary_df = pd.DataFrame(hsj_summary_rows)
print("\n\n===================== HSJ SUMMARY DATAFRAME =====================\n")
display(hsj_metrics_summary_df)

hsj_metrics_summary_df




===================== HOPSKIPJUMP METRICS SUMMARY =====================



################################################################
MODEL: LogisticRegression
################################################################

CLEAN PERFORMANCE
----------------------------
Accuracy: 0.9164214243672749
Precision: 0.9191983360683207
Recall: 0.9164214243672749
F1 Score: 0.9175247607419302

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699

Confusion Matrix:
[[1268   84]
 [  58  289]]

HOPSKIPJUMP ADVERSARIAL RESULTS
----------------------------
Clean Accuracy (HSJ dict): 0.91
Adv Accuracy   (HSJ):      0.675
Accuracy Drop: 0.235


##########################################################

,Model,Clean Accuracy,Clean F1,HSJ Adv Accuracy,HSJ Drop
0,LogisticRegression,0.916421,0.917525,0.675,0.235
1,NeuralNet,0.945851,0.945674,0.735,0.225
2,RandomForest,0.945262,0.943670,0.310,0.650
3,SVM,0.929880,0.925855,0.170,0.730
4,XGBoost,0.943496,0.942108,0.215,0.750


,Model,Clean Accuracy,Clean F1,HSJ Adv Accuracy,HSJ Drop
0,LogisticRegression,0.916421,0.917525,0.675,0.235
1,NeuralNet,0.945851,0.945674,0.735,0.225
2,RandomForest,0.945262,0.943670,0.310,0.650
3,SVM,0.929880,0.925855,0.170,0.730
4,XGBoost,0.943496,0.942108,0.215,0.750


In [18]:
from art.attacks.evasion import HopSkipJump
from art.estimators.classification import SklearnClassifier
from sklearn.metrics import accuracy_score
import numpy as np

def evaluate_robustness_hsj(model, X_test, y_test, max_samples=100, hsj_kwargs=None):
    if hsj_kwargs is None:
        hsj_kwargs = dict(
            max_iter=20,
            max_eval=10000,
            init_eval=100,
            init_size=10,
            targeted=False,
            norm=2,
        )

    # to numpy
    X = X_test.to_numpy().astype(np.float32) if hasattr(X_test, "to_numpy") else np.asarray(X_test, dtype=np.float32)
    y_raw = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)

    # encode to 0..K-1 for ART
    classes, y_int = np.unique(y_raw, return_inverse=True)

    n = min(max_samples, len(X))
    X = X[:n]
    y = y_int[:n]

    clip_values = (float(X.min()), float(X.max()))
    art_clf = SklearnClassifier(model=model, clip_values=clip_values)

    # clean accuracy on this subset
    preds_clean = np.argmax(art_clf.predict(X), axis=1)
    clean_acc = accuracy_score(y, preds_clean)

    # run HSJ
    hsj = HopSkipJump(classifier=art_clf, **hsj_kwargs)
    X_adv = hsj.generate(x=X, y=y)

    preds_adv = np.argmax(art_clf.predict(X_adv), axis=1)
    adv_acc = accuracy_score(y, preds_adv)

    return {
        "clean_acc": clean_acc,
        "adv_acc": adv_acc,
        "acc_drop": clean_acc - adv_acc,
        "num_samples": n,
    }
    


In [19]:
results = []
for name, (model, X_test, y_test) in model_runs.items():
    # optionally skip XGB if HSJ is unstable
    if "xgb" in name.lower():
        print(f"Skipping {name} (XGBoost) for HSJ robustness evaluation.")
        continue

    print(f"\nEvaluating robustness for {name}")
    res = evaluate_robustness_hsj(model, X_test, y_test, max_samples=100)
    res["Model"] = name
    print(res)
    results.append(res)

robust_df = pd.DataFrame(results)
display(robust_df)



Evaluating robustness for LogisticRegression


HopSkipJump: 100%|██████████| 100/100 [00:01<00:00, 56.59it/s]


{'clean_acc': 0.93, 'adv_acc': 0.69, 'acc_drop': 0.2400000000000001, 'num_samples': 100, 'Model': 'LogisticRegression'}

Evaluating robustness for NeuralNet


HopSkipJump: 100%|██████████| 100/100 [00:02<00:00, 49.79it/s]


{'clean_acc': 0.96, 'adv_acc': 0.71, 'acc_drop': 0.25, 'num_samples': 100, 'Model': 'NeuralNet'}

Evaluating robustness for RandomForest


HopSkipJump: 100%|██████████| 100/100 [08:38<00:00,  5.18s/it]


{'clean_acc': 0.96, 'adv_acc': 0.2, 'acc_drop': 0.76, 'num_samples': 100, 'Model': 'RandomForest'}

Evaluating robustness for SVM


HopSkipJump: 100%|██████████| 100/100 [00:06<00:00, 16.48it/s]

{'clean_acc': 0.91, 'adv_acc': 0.12, 'acc_drop': 0.79, 'num_samples': 100, 'Model': 'SVM'}
Skipping XGBoost (XGBoost) for HSJ robustness evaluation.


,clean_acc,adv_acc,acc_drop,num_samples,Model
0,0.93,0.69,0.24,100,LogisticRegression
1,0.96,0.71,0.25,100,NeuralNet
2,0.96,0.20,0.76,100,RandomForest
3,0.91,0.12,0.79,100,SVM
